# NeuroFusionAI: Step 2 - Model Training & Evaluation
This notebook trains and evaluates models for each individual modality, and then trains the multimodal fusion network.
Specifically:
1. **Image Classifier**: Fine-tunes ResNet-18 on drawings.
1.5 **MRI Classifier**: Fine-tunes EfficientNet-B0 on MRI scans.
2. **Voice Classifier**: Trains an MLP (PyTorch) and XGBoost Classifier on voice features.
3. **Telemonitoring Regressor**: Trains an MLP (PyTorch) and XGBoost Regressor on UPDRS scores.
4. **Multimodal Fusion**: Trains a late/mid-fusion network using combined embeddings.


In [9]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, mean_squared_error

# Resolve project root (walk up until we find the 'datasets' directory)
PROJECT_ROOT = os.path.abspath('.')
for p in [os.path.abspath(x) for x in sys.path if x]:
    if os.path.isdir(os.path.join(p, 'datasets')):
        PROJECT_ROOT = p
        break
    elif os.path.isdir(os.path.join(os.path.dirname(p), 'datasets')):
        PROJECT_ROOT = os.path.dirname(p)
        break
while not os.path.isdir(os.path.join(PROJECT_ROOT, 'datasets')) and PROJECT_ROOT != os.path.dirname(PROJECT_ROOT):
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Helper to resolve paths relative to project root
def P(*parts):
    return os.path.join(PROJECT_ROOT, *parts)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(P('outputs', 'checkpoints'), exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Device:', device)


Project root: d:\Parkison DATASET TRAINING | Device: cpu


## Reusable Fault-Tolerant PyTorchTrainer Class
We implement a research-grade `PyTorchTrainer` with support for automated checkpoint saving/loading, metrics CSV tracking, TensorBoard logging, early stopping, and automatic learning rate plateaus.


In [10]:
import time
from datetime import datetime
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_AVAILABLE = True
except ImportError:
    TENSORBOARD_AVAILABLE = False

class PyTorchTrainer:
    def __init__(
        self,
        model,
        criterion,
        optimizer,
        scheduler,
        device,
        model_name,
        metric_name='Accuracy',
        metric_mode='max',
        early_stopping_patience=15,
        is_regression=False
    ):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.model_name = model_name
        self.metric_name = metric_name
        self.metric_mode = metric_mode
        self.early_stopping_patience = early_stopping_patience
        self.is_regression = is_regression
        
        # Paths
        self.checkpoint_dir = P('outputs', 'checkpoints', self.model_name)
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        self.log_dir = P('outputs', 'logs')
        os.makedirs(self.log_dir, exist_ok=True)
        self.report_dir = P('outputs', 'reports')
        os.makedirs(self.report_dir, exist_ok=True)
        
        self.history_csv = P('outputs', 'logs', f'{self.model_name}_history.csv')
        self.metrics_csv = P('outputs', 'reports', f'{self.model_name}_metrics.csv')
        
        # Tensorboard
        if TENSORBOARD_AVAILABLE:
            self.tb_writer = SummaryWriter(log_dir=P('outputs', 'runs', self.model_name))
        else:
            self.tb_writer = None
        
        # State
        self.start_epoch = 1
        self.best_metric = -float('inf') if metric_mode == 'max' else float('inf')
        self.history = []
        self.patience_counter = 0
        
        # Try to resume
        self._load_checkpoint()
        
    def _load_checkpoint(self):
        latest_path = os.path.join(self.checkpoint_dir, 'latest.pth')
        if os.path.exists(latest_path):
            try:
                checkpoint = torch.load(latest_path, map_location=self.device)
                self.model.load_state_dict(checkpoint['model_state_dict'])
                self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
                if self.scheduler and checkpoint.get('scheduler_state_dict'):
                    self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
                self.start_epoch = checkpoint['epoch'] + 1
                self.best_metric = checkpoint['best_metric']
                self.history = checkpoint.get('history', [])
                self.patience_counter = checkpoint.get('patience_counter', 0)
                print(f'Checkpoint found for {self.model_name}.')
                print(f'Resuming from Epoch {self.start_epoch}...')
            except Exception as e:
                print(f'Failed to load checkpoint for {self.model_name}: {e}. Starting from epoch 1.')
        else:
            print(f'No checkpoint found for {self.model_name}. Starting from epoch 1.')
            
    def _save_checkpoint(self, epoch, val_metric, is_best):
        checkpoint = {
            'model_state_dict': {k: v.cpu().clone() for k, v in self.model.state_dict().items()},
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None,
            'epoch': epoch,
            'best_metric': self.best_metric,
            'history': self.history,
            'patience_counter': self.patience_counter
        }
        # Save latest
        torch.save(checkpoint, os.path.join(self.checkpoint_dir, 'latest.pth'))
        # Save epoch
        torch.save(checkpoint, os.path.join(self.checkpoint_dir, f'epoch_{epoch:03d}.pth'))
        # Save best
        if is_best:
            torch.save(checkpoint, os.path.join(self.checkpoint_dir, 'best.pth'))
            # Copy to legacy path for compatibility with explanation / inference scripts
            torch.save(checkpoint['model_state_dict'], P('outputs', 'checkpoints', f'{self.model_name}_best.pth'))
            
    def _is_improved(self, current, best):
        if self.metric_mode == 'max':
            return current > best
        else:
            return current < best
            
    def fit(self, train_loader, val_loader, total_epochs):
        if self.start_epoch > total_epochs:
            print(f'Model {self.model_name} already trained for {self.start_epoch - 1} epochs. Skipping.')
            return
        for epoch in range(self.start_epoch, total_epochs + 1):
            # Train
            self.model.train()
            train_loss = 0.0
            train_total = 0
            train_correct = 0
            
            for batch in train_loader:
                if len(batch) == 2:
                    feats, targets = batch
                    feats, targets = feats.to(self.device), targets.to(self.device)
                    outputs = self.model(feats)
                elif len(batch) == 5:
                    img_emb, mri_emb, voice_feat, clinical_feat, targets = batch
                    img_emb = img_emb.to(self.device)
                    mri_emb = mri_emb.to(self.device)
                    voice_feat = voice_feat.to(self.device)
                    clinical_feat = clinical_feat.to(self.device)
                    targets = targets.to(self.device)
                    outputs = self.model(img_emb, mri_emb, voice_feat, clinical_feat)
                else:
                    raise ValueError(f'Unknown batch format with {len(batch)} items.')
                    
                self.optimizer.zero_grad()
                loss = self.criterion(outputs, targets)
                loss.backward()
                self.optimizer.step()
                
                train_loss += loss.item() * targets.size(0)
                train_total += targets.size(0)
                
                if not self.is_regression:
                    _, preds = outputs.max(1)
                    train_correct += preds.eq(targets).sum().item()
                    
            train_epoch_loss = train_loss / train_total
            train_epoch_acc = (train_correct / train_total) if not self.is_regression else 0.0
            
            # Val
            self.model.eval()
            val_loss = 0.0
            val_total = 0
            val_preds = []
            val_targets = []
            
            with torch.no_grad():
                for batch in val_loader:
                    if len(batch) == 2:
                        feats, targets = batch
                        feats, targets = feats.to(self.device), targets.to(self.device)
                        outputs = self.model(feats)
                    elif len(batch) == 5:
                        img_emb, mri_emb, voice_feat, clinical_feat, targets = batch
                        img_emb = img_emb.to(self.device)
                        mri_emb = mri_emb.to(self.device)
                        voice_feat = voice_feat.to(self.device)
                        clinical_feat = clinical_feat.to(self.device)
                        targets = targets.to(self.device)
                        outputs = self.model(img_emb, mri_emb, voice_feat, clinical_feat)
                        
                    loss = self.criterion(outputs, targets)
                    val_loss += loss.item() * targets.size(0)
                    val_total += targets.size(0)
                    
                    if self.is_regression:
                        val_preds.append(outputs.cpu().numpy())
                        val_targets.append(targets.cpu().numpy())
                    else:
                        _, preds = outputs.max(1)
                        val_preds.extend(preds.cpu().numpy())
                        val_targets.extend(targets.cpu().numpy())
                        
            val_epoch_loss = val_loss / val_total
            
            # Compute metrics
            metrics = {}
            if self.is_regression:
                val_preds = np.concatenate(val_preds, axis=0)
                val_targets = np.concatenate(val_targets, axis=0)
                val_mse = mean_squared_error(val_targets, val_preds)
                val_r2 = r2_score(val_targets, val_preds)
                
                metrics['val_metric'] = val_r2 if self.metric_name == 'R2 Score' else val_mse
                metrics['MSE'] = val_mse
                metrics['R2 Score'] = val_r2
                metrics['Accuracy'] = np.nan
                metrics['Precision'] = np.nan
                metrics['Recall'] = np.nan
                metrics['F1 Score'] = np.nan
            else:
                val_preds = np.array(val_preds)
                val_targets = np.array(val_targets)
                val_acc = accuracy_score(val_targets, val_preds)
                val_prec = precision_score(val_targets, val_preds, zero_division=0)
                val_rec = recall_score(val_targets, val_preds, zero_division=0)
                val_f1 = f1_score(val_targets, val_preds, zero_division=0)
                
                metrics['val_metric'] = val_acc
                metrics['Accuracy'] = val_acc
                metrics['Precision'] = val_prec
                metrics['Recall'] = val_rec
                metrics['F1 Score'] = val_f1
                metrics['R2 Score'] = np.nan
                metrics['MSE'] = np.nan
                
            current_metric = metrics['val_metric']
            
            # Scheduler step
            if self.scheduler:
                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(current_metric)
                else:
                    self.scheduler.step()
                    
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Check improvement
            is_best = self._is_improved(current_metric, self.best_metric)
            status_str = 'No Improvement'
            if is_best:
                self.best_metric = current_metric
                self.patience_counter = 0
                status_str = '✅ New Best Model Saved'
            else:
                self.patience_counter += 1
                
            # Log state
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            self.history.append({
                'Timestamp': timestamp,
                'Epoch': epoch,
                'Train Loss': train_epoch_loss,
                'Val Loss': val_epoch_loss,
                self.metric_name: current_metric,
                f'Best {self.metric_name}': self.best_metric
            })
            
            # Save Checkpoints
            self._save_checkpoint(epoch, current_metric, is_best)
            
            # Write to training_history.csv
            history_df = pd.DataFrame(self.history)
            history_df.to_csv(self.history_csv, index=False)
            
            # Write to metrics.csv
            epoch_metrics = {
                'Epoch': epoch,
                'Train Loss': train_epoch_loss,
                'Validation Loss': val_epoch_loss,
                'Accuracy': metrics['Accuracy'],
                'Precision': metrics['Precision'],
                'Recall': metrics['Recall'],
                'F1 Score': metrics['F1 Score'],
                'Learning Rate': current_lr,
                'Timestamp': timestamp
            }
            if self.is_regression:
                epoch_metrics['R2 Score'] = metrics['R2 Score']
                epoch_metrics['MSE'] = metrics['MSE']
                
            if os.path.exists(self.metrics_csv):
                try:
                    metrics_df = pd.read_csv(self.metrics_csv)
                    metrics_df = pd.concat([metrics_df, pd.DataFrame([epoch_metrics])], ignore_index=True)
                except Exception:
                    metrics_df = pd.DataFrame([epoch_metrics])
            else:
                metrics_df = pd.DataFrame([epoch_metrics])
            metrics_df.to_csv(self.metrics_csv, index=False)
            
            # TensorBoard logging
            if self.tb_writer is not None:
                self.tb_writer.add_scalar('Loss/train', train_epoch_loss, epoch)
                self.tb_writer.add_scalar('Loss/val', val_epoch_loss, epoch)
                self.tb_writer.add_scalar('Learning_Rate', current_lr, epoch)
                if self.is_regression:
                    self.tb_writer.add_scalar('Metric/val_R2', metrics['R2 Score'], epoch)
                    self.tb_writer.add_scalar('Metric/val_MSE', metrics['MSE'], epoch)
                else:
                    self.tb_writer.add_scalar('Metric/val_Accuracy', metrics['Accuracy'], epoch)
                    self.tb_writer.add_scalar('Metric/val_Precision', metrics['Precision'], epoch)
                    self.tb_writer.add_scalar('Metric/val_Recall', metrics['Recall'], epoch)
                    self.tb_writer.add_scalar('Metric/val_F1', metrics['F1 Score'], epoch)
                
            # Live Console Output matching the exact format
            print('===================================================')
            print(f'Epoch {epoch} / {total_epochs}')
            print()
            print(f'Train Loss      : {train_epoch_loss:.4f}')
            print(f'Validation Loss : {val_epoch_loss:.4f}')
            print()
            if self.is_regression:
                print(f'R^2 Score        : {metrics["R2 Score"]:.4f}')
                print(f'Best R^2 Score   : {self.best_metric:.4f}')
            else:
                print(f'Accuracy         : {metrics["Accuracy"]*100:.2f}%')
                print(f'Best Accuracy    : {self.best_metric*100:.2f}%')
            print()
            print(f'Learning Rate    : {current_lr:.6f}')
            print()
            print(f'Status           : {status_str}')
            print()
            print(f'Time             : {timestamp}')
            print('===================================================')
            sys.stdout.flush()
            
            # Early stopping check
            if self.patience_counter >= self.early_stopping_patience:
                print(f'Early stopping triggered at epoch {epoch}. No improvement for {self.early_stopping_patience} epochs.')
                break
                
        if self.tb_writer is not None:
            self.tb_writer.close()


## 1. Train Drawings Classifier (Image Model)
We load the drawing images, instantiate the `ImageDrawingClassifier` model, and train using our `PyTorchTrainer`.


In [11]:
from preprocessing.image_preprocessing import get_image_dataloader
from models.image_model import ImageDrawingClassifier

train_loader = get_image_dataloader('train', batch_size=16, shuffle=True, augment=True)
val_loader = get_image_dataloader('validation', batch_size=16, shuffle=False)

model = ImageDrawingClassifier(num_classes=2, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.resnet.fc.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

trainer = PyTorchTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    model_name='image',
    metric_name='Accuracy',
    metric_mode='max',
    early_stopping_patience=10
)

trainer.fit(train_loader, val_loader, total_epochs=10)


Checkpoint found for image.
Resuming from Epoch 11...
Model image already trained for 10 epochs. Skipping.


## 1.1 Train MRI Classifier (Neuroimaging Model)
We fine-tune the pretrained EfficientNet-B0 model on the preprocessed MRI images.


In [12]:
from preprocessing.mri_preprocessing import get_mri_dataloader
from models.mri_model import MRIClassifier

mri_train_loader = get_mri_dataloader('train', batch_size=16, shuffle=True, augment=True)
mri_val_loader = get_mri_dataloader('validation', batch_size=16, shuffle=False)

mri_model = MRIClassifier(num_classes=2, pretrained=True).to(device)
mri_criterion = nn.CrossEntropyLoss()
mri_optimizer = optim.Adam(mri_model.efficientnet.classifier.parameters(), lr=0.001)
mri_scheduler = optim.lr_scheduler.ReduceLROnPlateau(mri_optimizer, mode='max', factor=0.5, patience=3)

mri_trainer = PyTorchTrainer(
    model=mri_model,
    criterion=mri_criterion,
    optimizer=mri_optimizer,
    scheduler=mri_scheduler,
    device=device,
    model_name='mri',
    metric_name='Accuracy',
    metric_mode='max',
    early_stopping_patience=10
)

mri_trainer.fit(mri_train_loader, mri_val_loader, total_epochs=10)


Checkpoint found for mri.
Resuming from Epoch 11...
Model mri already trained for 10 epochs. Skipping.


## 2. Train Voice Classifier (Acoustics Model)
We evaluate both PyTorch MLP and XGBoost models on the Oxford voice dataset.


In [13]:
from preprocessing.voice_preprocessing import get_voice_dataloader
from models.voice_model import VoiceMLPClassifier, VoiceXGBClassifier

voice_train_loader = get_voice_dataloader('train', batch_size=16, shuffle=True)
voice_val_loader = get_voice_dataloader('validation', batch_size=16, shuffle=False)

# A. PyTorch MLP Classifier
voice_model = VoiceMLPClassifier(input_dim=22, hidden_dim=64, num_classes=2).to(device)
voice_criterion = nn.CrossEntropyLoss()
voice_opt = optim.Adam(voice_model.parameters(), lr=0.005, weight_decay=1e-4)
voice_scheduler = optim.lr_scheduler.ReduceLROnPlateau(voice_opt, mode='max', factor=0.5, patience=5)

voice_trainer = PyTorchTrainer(
    model=voice_model,
    criterion=voice_criterion,
    optimizer=voice_opt,
    scheduler=voice_scheduler,
    device=device,
    model_name='voice_mlp',
    metric_name='Accuracy',
    metric_mode='max',
    early_stopping_patience=15
)

voice_trainer.fit(voice_train_loader, voice_val_loader, total_epochs=20)

# B. XGBoost Classifier
print('\nTraining Voice XGBoost Classifier...')
train_df = pd.read_csv(P('datasets', 'train', 'voice', 'oxford_train.csv'))
val_df = pd.read_csv(P('datasets', 'validation', 'voice', 'oxford_validation.csv'))

X_train = train_df.drop(columns=['status']).values
y_train = train_df['status'].values
X_val = val_df.drop(columns=['status']).values
y_val = val_df['status'].values

xgb_model = VoiceXGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_val)

xgb_acc = accuracy_score(y_val, xgb_preds)
xgb_precision = precision_score(y_val, xgb_preds, zero_division=0)
xgb_recall = recall_score(y_val, xgb_preds, zero_division=0)
xgb_f1 = f1_score(y_val, xgb_preds, zero_division=0)

print('--- Voice XGBoost Results ---')
print(f'Validation Accuracy : {xgb_acc:.4f}')
print(f'Validation Precision: {xgb_precision:.4f}')
print(f'Validation Recall   : {xgb_recall:.4f}')
print(f'Validation F1-Score : {xgb_f1:.4f}')
xgb_model.save(P('outputs', 'checkpoints', 'voice_xgb_best.pkl'))
print('Saved Voice XGBoost Classifier model!')


Checkpoint found for voice_mlp.
Resuming from Epoch 21...
Model voice_mlp already trained for 20 epochs. Skipping.

Training Voice XGBoost Classifier...
--- Voice XGBoost Results ---
Validation Accuracy : 0.9655
Validation Precision: 0.9565
Validation Recall   : 1.0000
Validation F1-Score : 0.9778
Saved Voice XGBoost Classifier model!


c:\Users\Harsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:17:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## 3. Train Telemonitoring Regressor (UPDRS Severity Model)
We train a model to predict the motor and total UPDRS scores from telemonitoring.


In [14]:
from preprocessing.telemonitor_preprocessing import get_telemonitor_dataloader
from models.telemonitor_model import TelemonitorMLPRegressor, TelemonitorXGBRegressor

tele_train_loader = get_telemonitor_dataloader('train', batch_size=32, shuffle=True)
tele_val_loader = get_telemonitor_dataloader('validation', batch_size=32, shuffle=False)

# A. PyTorch MLP Regressor
tele_model = TelemonitorMLPRegressor(input_dim=19, hidden_dim=64, output_dim=2).to(device)
tele_criterion = nn.MSELoss()
tele_opt = optim.Adam(tele_model.parameters(), lr=0.005, weight_decay=1e-4)
tele_scheduler = optim.lr_scheduler.ReduceLROnPlateau(tele_opt, mode='max', factor=0.5, patience=5)

tele_trainer = PyTorchTrainer(
    model=tele_model,
    criterion=tele_criterion,
    optimizer=tele_opt,
    scheduler=tele_scheduler,
    device=device,
    model_name='telemonitor_mlp',
    metric_name='R2 Score',
    metric_mode='max',
    early_stopping_patience=15,
    is_regression=True
)

tele_trainer.fit(tele_train_loader, tele_val_loader, total_epochs=20)

# B. XGBoost Regressor
print('\nTraining Telemonitoring XGBoost Regressor...')
train_tele_df = pd.read_csv(P('datasets', 'train', 'telemonitoring', 'telemonitor_train.csv'))
val_tele_df = pd.read_csv(P('datasets', 'validation', 'telemonitoring', 'telemonitor_validation.csv'))

X_train_t = train_tele_df.drop(columns=['motor_UPDRS', 'total_UPDRS']).values
y_train_t = train_tele_df[['motor_UPDRS', 'total_UPDRS']].values
X_val_t = val_tele_df.drop(columns=['motor_UPDRS', 'total_UPDRS']).values
y_val_t = val_tele_df[['motor_UPDRS', 'total_UPDRS']].values

xgb_reg = TelemonitorXGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05)
xgb_reg.fit(X_train_t, y_train_t)
xgb_preds_t = xgb_reg.predict(X_val_t)

xgb_r2 = r2_score(y_val_t, xgb_preds_t)
xgb_mse = mean_squared_error(y_val_t, xgb_preds_t)

print('--- Telemonitor XGBoost Results ---')
print(f'Validation MSE     : {xgb_mse:.4f}')
print(f'Validation R^2 Score: {xgb_r2:.4f}')
xgb_reg.save(P('outputs', 'checkpoints', 'telemonitor_xgb_best.pkl'))
print('Saved Telemonitor XGBoost Regressor model!')


Checkpoint found for telemonitor_mlp.
Resuming from Epoch 21...
Model telemonitor_mlp already trained for 20 epochs. Skipping.

Training Telemonitoring XGBoost Regressor...
--- Telemonitor XGBoost Results ---
Validation MSE     : 18.4736
Validation R^2 Score: 0.7811
Saved Telemonitor XGBoost Regressor model!


## 4. Train Multimodal Fusion Network
We train the joint classifier using the projection head design on fused representations.


In [15]:
from preprocessing.fusion_preprocessing import get_fusion_dataloader
from models.fusion_model import MultimodalClassifier

fusion_train_loader = get_fusion_dataloader('train', batch_size=16)
fusion_val_loader = get_fusion_dataloader('validation', batch_size=16)

fusion_model = MultimodalClassifier(image_dim=256, mri_dim=256, voice_dim=22, clinical_dim=19, fusion_dim=32, num_classes=2).to(device)
fusion_criterion = nn.CrossEntropyLoss()
fusion_opt = optim.Adam(fusion_model.parameters(), lr=0.005)
fusion_scheduler = optim.lr_scheduler.ReduceLROnPlateau(fusion_opt, mode='max', factor=0.5, patience=5)

fusion_trainer = PyTorchTrainer(
    model=fusion_model,
    criterion=fusion_criterion,
    optimizer=fusion_opt,
    scheduler=fusion_scheduler,
    device=device,
    model_name='fusion',
    metric_name='Accuracy',
    metric_mode='max',
    early_stopping_patience=15
)

fusion_trainer.fit(fusion_train_loader, fusion_val_loader, total_epochs=15)


Checkpoint found for fusion.
Resuming from Epoch 16...
Model fusion already trained for 15 epochs. Skipping.


## 5. Final Model Selection & Evaluation (Test Set)
We load the `best.pth` checkpoints (for PyTorch models) or pickled models (for XGBoost models) and evaluate them on the test set.


In [16]:
from datetime import datetime
print('Evaluating all models on the Test Set...\n')

# A. Drawings Classifier (Image Model)
from preprocessing.image_preprocessing import get_image_dataloader
from models.image_model import ImageDrawingClassifier
image_model = ImageDrawingClassifier(num_classes=2).to(device)
image_model.load_state_dict(torch.load(P('outputs', 'checkpoints', 'image', 'best.pth'), map_location=device)['model_state_dict'])
image_model.eval()
image_test_loader = get_image_dataloader('test', batch_size=16, shuffle=False)

img_preds, img_targets = [], []
with torch.no_grad():
    for imgs, labels in image_test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = image_model(imgs)
        _, preds = outputs.max(1)
        img_preds.extend(preds.cpu().numpy())
        img_targets.extend(labels.cpu().numpy())

img_acc = accuracy_score(img_targets, img_preds)
img_prec = precision_score(img_targets, img_preds, zero_division=0)
img_rec = recall_score(img_targets, img_preds, zero_division=0)
img_f1 = f1_score(img_targets, img_preds, zero_division=0)

# B. MRI Classifier
from preprocessing.mri_preprocessing import get_mri_dataloader
from models.mri_model import MRIClassifier
mri_model = MRIClassifier(num_classes=2).to(device)
mri_model.load_state_dict(torch.load(P('outputs', 'checkpoints', 'mri', 'best.pth'), map_location=device)['model_state_dict'])
mri_model.eval()
mri_test_loader = get_mri_dataloader('test', batch_size=16, shuffle=False)

mri_preds, mri_targets = [], []
with torch.no_grad():
    for imgs, labels in mri_test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = mri_model(imgs)
        _, preds = outputs.max(1)
        mri_preds.extend(preds.cpu().numpy())
        mri_targets.extend(labels.cpu().numpy())

mri_acc = accuracy_score(mri_targets, mri_preds)
mri_prec = precision_score(mri_targets, mri_preds, zero_division=0)
mri_rec = recall_score(mri_targets, mri_preds, zero_division=0)
mri_f1 = f1_score(mri_targets, mri_preds, zero_division=0)

# C. Voice MLP Classifier
from preprocessing.voice_preprocessing import get_voice_dataloader
from models.voice_model import VoiceMLPClassifier
voice_mlp = VoiceMLPClassifier(input_dim=22, hidden_dim=64, num_classes=2).to(device)
voice_mlp.load_state_dict(torch.load(P('outputs', 'checkpoints', 'voice_mlp', 'best.pth'), map_location=device)['model_state_dict'])
voice_mlp.eval()
voice_test_loader = get_voice_dataloader('test', batch_size=16, shuffle=False)

vmlp_preds, vmlp_targets = [], []
with torch.no_grad():
    for feats, labels in voice_test_loader:
        feats, labels = feats.to(device), labels.to(device)
        outputs = voice_mlp(feats)
        _, preds = outputs.max(1)
        vmlp_preds.extend(preds.cpu().numpy())
        vmlp_targets.extend(labels.cpu().numpy())

vmlp_acc = accuracy_score(vmlp_targets, vmlp_preds)
vmlp_prec = precision_score(vmlp_targets, vmlp_preds, zero_division=0)
vmlp_rec = recall_score(vmlp_targets, vmlp_preds, zero_division=0)
vmlp_f1 = f1_score(vmlp_targets, vmlp_preds, zero_division=0)

# D. Voice XGBoost Classifier
from models.voice_model import VoiceXGBClassifier
voice_xgb = VoiceXGBClassifier()
voice_xgb.load(P('outputs', 'checkpoints', 'voice_xgb_best.pkl'))
test_voice_df = pd.read_csv(P('datasets', 'test', 'voice', 'oxford_test.csv'))
X_test_v = test_voice_df.drop(columns=['status']).values
y_test_v = test_voice_df['status'].values
vxgb_preds = voice_xgb.predict(X_test_v)

vxgb_acc = accuracy_score(y_test_v, vxgb_preds)
vxgb_prec = precision_score(y_test_v, vxgb_preds, zero_division=0)
vxgb_rec = recall_score(y_test_v, vxgb_preds, zero_division=0)
vxgb_f1 = f1_score(y_test_v, vxgb_preds, zero_division=0)

# E. Telemonitoring MLP Regressor
from preprocessing.telemonitor_preprocessing import get_telemonitor_dataloader
from models.telemonitor_model import TelemonitorMLPRegressor
tele_mlp = TelemonitorMLPRegressor(input_dim=19, hidden_dim=64, output_dim=2).to(device)
tele_mlp.load_state_dict(torch.load(P('outputs', 'checkpoints', 'telemonitor_mlp', 'best.pth'), map_location=device)['model_state_dict'])
tele_mlp.eval()
tele_test_loader = get_telemonitor_dataloader('test', batch_size=32, shuffle=False)

tmlp_preds, tmlp_targets = [], []
with torch.no_grad():
    for feats, targets in tele_test_loader:
        feats, targets = feats.to(device), targets.to(device)
        outputs = tele_mlp(feats)
        tmlp_preds.append(outputs.cpu().numpy())
        tmlp_targets.append(targets.cpu().numpy())

tmlp_preds = np.concatenate(tmlp_preds, axis=0)
tmlp_targets = np.concatenate(tmlp_targets, axis=0)
tmlp_mse = mean_squared_error(tmlp_targets, tmlp_preds)
tmlp_r2 = r2_score(tmlp_targets, tmlp_preds)

# F. Telemonitoring XGBoost Regressor
from models.telemonitor_model import TelemonitorXGBRegressor
tele_xgb = TelemonitorXGBRegressor()
tele_xgb.load(P('outputs', 'checkpoints', 'telemonitor_xgb_best.pkl'))
test_tele_df = pd.read_csv(P('datasets', 'test', 'telemonitoring', 'telemonitor_test.csv'))
X_test_t = test_tele_df.drop(columns=['motor_UPDRS', 'total_UPDRS']).values
y_test_t = test_tele_df[['motor_UPDRS', 'total_UPDRS']].values
txgb_preds = tele_xgb.predict(X_test_t)

txgb_mse = mean_squared_error(y_test_t, txgb_preds)
txgb_r2 = r2_score(y_test_t, txgb_preds)

# G. Multimodal Fusion Classifier
from preprocessing.fusion_preprocessing import get_fusion_dataloader
from models.fusion_model import MultimodalClassifier
fusion_model = MultimodalClassifier(image_dim=256, mri_dim=256, voice_dim=22, clinical_dim=19, fusion_dim=32).to(device)
fusion_model.load_state_dict(torch.load(P('outputs', 'checkpoints', 'fusion', 'best.pth'), map_location=device)['model_state_dict'])
fusion_model.eval()
fusion_test_loader = get_fusion_dataloader('test', batch_size=16)

fus_preds, fus_targets = [], []
with torch.no_grad():
    for img_emb, mri_emb, voice_feat, clinical_feat, label in fusion_test_loader:
        img_emb = img_emb.to(device)
        mri_emb = mri_emb.to(device)
        voice_feat = voice_feat.to(device)
        clinical_feat = clinical_feat.to(device)
        label = label.to(device)
        outputs = fusion_model(img_emb, mri_emb, voice_feat, clinical_feat)
print('Multimodal Fusion model saved successfully!')


Evaluating all models on the Test Set...



c:\Users\Harsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Harsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Harsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Harsh\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  s

Multimodal Fusion model saved successfully!
